# Fine-Tune Legal-HeBERT on Trial Court Verdicts

This notebook fine-tunes `avichr/Legal-heBERT` on your 100K trial court verdicts
using **Masked Language Modeling (MLM)** — teaching the model the language patterns
of trial courts (testimonies, cross-examinations, protocols) which differ from
Supreme Court decisions.

**Requirements:**
- Google Colab with GPU (Runtime → Change runtime type → T4 GPU)
- Verdict files uploaded to Google Drive

**Output:**
- Fine-tuned model pushed to HuggingFace Hub
- Ready to deploy on Railway via `LEGAL_HEBERT_MODEL` env var

## Step 1: Setup

In [ ]:
!pip install -q transformers datasets accelerate huggingface_hub torch

In [ ]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU! Go to Runtime → Change runtime type → T4 GPU")

## Step 2: Mount Google Drive

Upload your verdict `.txt` files to a folder in Google Drive.
Update the path below to point to that folder.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# CONFIGURE THIS: Path to your verdict files in Google Drive
# ============================================================
VERDICTS_FOLDER = "/content/drive/MyDrive/verdicts"  # <-- CHANGE THIS

# Your HuggingFace username (for pushing the model)
HF_USERNAME = "your-username"  # <-- CHANGE THIS

# Model name on HuggingFace Hub
MODEL_NAME = "Legal-heBERT-trial-courts"
# ============================================================

import os
import glob

# Find all text files
patterns = ["*.txt", "*.TXT"]
all_files = []
for pattern in patterns:
    all_files.extend(glob.glob(os.path.join(VERDICTS_FOLDER, "**", pattern), recursive=True))

print(f"Found {len(all_files):,} verdict files")
if all_files:
    # Show sample
    sample = all_files[0]
    with open(sample, 'r', encoding='utf-8', errors='ignore') as f:
        text = f.read(500)
    print(f"\nSample from: {os.path.basename(sample)}")
    print(text[:500])

## Step 3: Prepare Dataset

Read all files and create a HuggingFace Dataset for training.
Texts are chunked into 512-token blocks for BERT.

In [ ]:
from transformers import AutoTokenizer

BASE_MODEL = "avichr/Legal-heBERT"
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

print(f"Tokenizer loaded: {BASE_MODEL}")
print(f"Vocab size: {tokenizer.vocab_size:,}")

In [ ]:
import re
from datasets import Dataset

def clean_text(text):
    """Basic cleaning for legal text."""
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text)
    # Remove page markers
    text = re.sub(r'---\s*עמוד\s*\d+\s*---', ' ', text)
    # Normalize quotes
    text = text.replace('\u05f4', '"').replace('\u05f3', "'")
    return text.strip()

# Read all files
print("Reading verdict files...")
all_texts = []
errors = 0

for i, filepath in enumerate(all_files):
    if i % 10000 == 0:
        print(f"  {i:,}/{len(all_files):,} files read...")
    try:
        with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
            text = f.read()
        text = clean_text(text)
        # Skip very short files (< 100 chars) — probably empty/corrupt
        if len(text) >= 100:
            all_texts.append(text)
    except Exception:
        errors += 1

print(f"\nLoaded {len(all_texts):,} verdicts ({errors} errors)")
total_chars = sum(len(t) for t in all_texts)
print(f"Total text: {total_chars / 1e9:.2f} GB")

In [ ]:
# Tokenize and chunk into 512-token blocks
MAX_LENGTH = 512

print("Tokenizing all texts (this takes a few minutes)...")

all_input_ids = []
for i, text in enumerate(all_texts):
    if i % 10000 == 0:
        print(f"  Tokenizing {i:,}/{len(all_texts):,}...")
    tokens = tokenizer(text, add_special_tokens=False, truncation=False)["input_ids"]
    # Split into chunks of MAX_LENGTH
    for j in range(0, len(tokens) - MAX_LENGTH + 1, MAX_LENGTH):
        chunk = tokens[j:j + MAX_LENGTH]
        all_input_ids.append(chunk)

print(f"\nCreated {len(all_input_ids):,} training chunks of {MAX_LENGTH} tokens")

# Create dataset
dataset = Dataset.from_dict({"input_ids": all_input_ids})

# Split into train/val (97/3)
split = dataset.train_test_split(test_size=0.03, seed=42)
train_dataset = split["train"]
val_dataset = split["test"]

print(f"Train: {len(train_dataset):,} chunks")
print(f"Validation: {len(val_dataset):,} chunks")

## Step 4: Fine-Tune with MLM

Masked Language Modeling: randomly mask 15% of tokens and train the model
to predict them. This teaches BERT the language of trial courts.

In [ ]:
from transformers import (
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
)

# Load base model
model = AutoModelForMaskedLM.from_pretrained(BASE_MODEL)
print(f"Model parameters: {model.num_parameters() / 1e6:.1f}M")

# MLM data collator — masks 15% of tokens randomly
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15,
)

In [ ]:
# Training configuration — optimized for Colab T4 (16GB VRAM)
OUTPUT_DIR = "/content/legal_hebert_finetuned"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,

    # Training
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,

    # Evaluation
    eval_strategy="steps",
    eval_steps=2000,
    save_steps=2000,
    save_total_limit=3,

    # Performance
    fp16=True,  # Mixed precision — faster + less memory
    gradient_accumulation_steps=2,  # Effective batch = 32
    dataloader_num_workers=2,

    # Logging
    logging_steps=500,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print("Ready to train!")
print(f"Estimated training time on T4: ~{len(train_dataset) * 3 / 16 / 3600 * 1.5:.1f} hours")

In [ ]:
# TRAIN!
print("Starting training...")
trainer.train()
print("\nTraining complete!")

In [ ]:
# Evaluate
results = trainer.evaluate()
print(f"Validation loss: {results['eval_loss']:.4f}")
print(f"Perplexity: {2.718 ** results['eval_loss']:.2f}")

## Step 5: Push to HuggingFace Hub

The model will be uploaded to your HuggingFace account.
Then set `LEGAL_HEBERT_MODEL=your-username/Legal-heBERT-trial-courts` in Railway.

In [ ]:
from huggingface_hub import notebook_login

# Login to HuggingFace (get token from https://huggingface.co/settings/tokens)
notebook_login()

In [ ]:
# Push to Hub
repo_name = f"{HF_USERNAME}/{MODEL_NAME}"

print(f"Pushing model to: https://huggingface.co/{repo_name}")
trainer.push_to_hub(repo_name)
tokenizer.push_to_hub(repo_name)

print(f"\n{'='*60}")
print(f"DONE! Model available at:")
print(f"https://huggingface.co/{repo_name}")
print(f"\nTo use in JETHRO, set this env var in Railway:")
print(f"  LEGAL_HEBERT_MODEL={repo_name}")
print(f"{'='*60}")

## Step 6: Quick Test

Test the fine-tuned model on a legal sentence with a masked token.

In [ ]:
from transformers import pipeline

# Test with MLM
fill_mask = pipeline("fill-mask", model=OUTPUT_DIR, tokenizer=OUTPUT_DIR)

test_sentences = [
    "העד הצהיר כי [MASK] את התשלום במועד",
    "הנתבע [MASK] בחקירתו הנגדית כי לא היה נוכח",
    "בית המשפט [MASK] את התביעה בהעדר ראיות",
    "התובע טען כי ההסכם [MASK] ללא הסכמתו",
]

print("=" * 60)
print("Fine-tuned model predictions:")
print("=" * 60)
for sent in test_sentences:
    results = fill_mask(sent)
    print(f"\n{sent}")
    for r in results[:3]:
        print(f"  → {r['token_str']:>10s}  ({r['score']:.3f})")

## Backup to Google Drive (Optional)

In [ ]:
import shutil

backup_path = "/content/drive/MyDrive/legal_hebert_finetuned"
print(f"Backing up model to Google Drive: {backup_path}")
shutil.copytree(OUTPUT_DIR, backup_path, dirs_exist_ok=True)
print("Backup complete!")